# Topic 32 · A Transformer from scratch: English → Spanish

CSCI 6379 · companion to [Transformers, Part II: the decoder](https://dongchul.kim/csci6379-fall2026/topic/32) and its [slides](https://dongchul.kim/csci6379/slides/topic32.pdf) (the Implementation section).

The whole encoder-decoder Transformer in about 120 lines of PyTorch: one attention class used three ways, the padding and causal masks,
encoder and decoder layers, training with teacher forcing, and translating one word at a time.

It runs on a **CPU in under a minute**; no GPU needed. **Runtime > Run all**, then change things and run again.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import matplotlib.pyplot as plt

PAD, SOS, EOS, UNK = 0, 1, 2, 3     # the four special token ids

## 1. One attention class for every attention

`d` is the vector size of a word, `h` the number of heads, and `dk = d // h` the size each head works with.
`split` cuts the `d` numbers of every word into `h` pieces of `dk`; the heads run side by side as one extra tensor dimension,
and `reshape` joins them back. The `mask` says who may look at whom (`True` = may look).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.q, self.k = nn.Linear(d, d), nn.Linear(d, d)
        self.v, self.o = nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x_q, x_kv, mask):   # mask: True = may look
        B, Tq, d = x_q.shape
        def split(t):                     # (B,T,d) -> (B,h,T,dk)
            return t.view(B, -1, self.h, self.dk).transpose(1, 2)
        Q = split(self.q(x_q))                    # ①
        K, V = split(self.k(x_kv)), split(self.v(x_kv))  # ②
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)  # ③
        scores = scores.masked_fill(~mask, float("-inf"))  # ④
        out = torch.softmax(scores, dim=-1) @ V   # ⑤⑥
        out = out.transpose(1, 2).reshape(B, Tq, d)  # ⑦
        return self.o(out)                        # ⑧

## 2. Two masks

* **Padding mask**: sentences in a batch have different lengths, so short ones are filled with `<pad>` (id 0). Nobody may look at a `<pad>`.
* **Causal mask**: in the decoder, position *t* may look only at positions up to *t* (and not at `<pad>`).

In [ ]:
def pad_mask(src):      # (B, 1, 1, Ts)
    return (src != PAD)[:, None, None, :]

def causal_mask(tgt):   # (B, 1, Tt, Tt)
    T = tgt.size(1)
    past = torch.ones(T, T, dtype=torch.bool).tril()
    return past & (tgt != PAD)[:, None, None, :]

In [ ]:
# What the causal mask looks like for a 5-token target whose last token is <pad>
tgt = torch.tensor([[SOS, 4, 18, EOS, PAD]])
print(causal_mask(tgt)[0, 0].int())

## 3. Encoder and decoder layers

Each sub-layer is followed by **Add & Norm**: `norm(x + sublayer(x))`. The `x +` is the residual.
The encoder layer has two sub-layers (self-attention, feed-forward), so two residuals; the decoder layer has three
(masked self-attention, cross-attention, feed-forward), so three. Cross-attention is where the decoder reads the encoder's output (`memory`).

In [ ]:
def feed_forward(d, d_ff):
    return nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(),
                         nn.Linear(d_ff, d))

class EncoderLayer(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d, h)
        self.ff = feed_forward(d, d_ff)
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x, src_mask):
        x = self.norm1(x + self.attn(x, x, src_mask))  # ①②
        return self.norm2(x + self.ff(x))              # ③④

class DecoderLayer(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.self_attn = MultiHeadAttention(d, h)
        self.cross = MultiHeadAttention(d, h)
        self.ff = feed_forward(d, d_ff)
        self.norm1, self.norm2, self.norm3 = (
            nn.LayerNorm(d) for _ in range(3))

    def forward(self, y, memory, tgt_mask, src_mask):
        y = self.norm1(y + self.self_attn(y, y, tgt_mask))   # ①②
        y = self.norm2(y + self.cross(y, memory, src_mask))  # ③④
        return self.norm3(y + self.ff(y))                    # ⑤⑥

## 4. The whole model

`nn.ModuleList` makes `N = n_layers` copies of each layer, each with its own weights, and the `for` loops pass the data through them in order.
`encode` runs once per sentence; `decode` returns a score (**logit**) for every Spanish word at every position.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d=128, h=4,
                 n_layers=2, d_ff=256, max_len=32):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d)   # ①
        self.tgt_emb = nn.Embedding(tgt_vocab, d)   # ④
        self.pos = nn.Embedding(max_len, d)   # ①④ positions
        L = range(n_layers)
        self.enc = nn.ModuleList(EncoderLayer(d, h, d_ff) for _ in L)  # ②
        self.dec = nn.ModuleList(DecoderLayer(d, h, d_ff) for _ in L)  # ⑤
        self.out = nn.Linear(d, tgt_vocab)    # ⑥ a score per word

    def embed(self, emb, ids):
        pos = torch.arange(ids.size(1))
        return emb(ids) + self.pos(pos)

    def encode(self, src):
        x = self.embed(self.src_emb, src)     # ①
        for layer in self.enc:                # ②
            x = layer(x, pad_mask(src))
        return x                              # ③ memory

    def decode(self, tgt, memory, src):
        y = self.embed(self.tgt_emb, tgt)     # ④
        for layer in self.dec:                # ⑤
            y = layer(y, memory, causal_mask(tgt), pad_mask(src))
        return self.out(y)                    # ⑥ (B, Tt, vocab)

    def forward(self, src, tgt):
        return self.decode(tgt, self.encode(src), src)

## 5. Data: words to ids, and the one-step shift

`encode` adds `<sos>` (1) at the start and `<eos>` (2) at the end; real words get ids from 4, in sorted order.
`make_batch` pads every sentence to the longest one with 0 = `<pad>`, and splits the Spanish side in two:
`tgt_in` (starts with `<sos>`) is what the decoder reads, and `tgt_out` (one step ahead) is what it must predict.

In [ ]:
pairs = [("hello", "hola"), ("how are you", "cómo estás"), ("good morning", "buenos días"),
         ("thank you", "gracias"), ("see you later", "hasta luego"), ("good night", "buenas noches"),
         ("please", "por favor"), ("yes", "sí"), ("no", "no"),
         ("what is your name", "cuál es tu nombre"), ("my name is", "mi nombre es")]

In [ ]:
class Vocab:
    def __init__(self, sentences):
        words = sorted({w for s in sentences for w in s.split()})
        self.itos = ["<pad>", "<sos>", "<eos>", "<unk>"] + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, s):
        ids = [self.stoi.get(w, UNK) for w in s.split()]
        return [SOS] + ids + [EOS]                # ①

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids)

en = Vocab(e for e, _ in pairs)       # sorted: same ids every run
es = Vocab(s for _, s in pairs)

def make_batch(pairs):
    src = [torch.tensor(en.encode(e)) for e, _ in pairs]
    tgt = [torch.tensor(es.encode(s)) for _, s in pairs]
    src = pad_sequence(src, batch_first=True)    # ② pad with 0
    tgt = pad_sequence(tgt, batch_first=True)    # ②
    return src, tgt[:, :-1], tgt[:, 1:]          # ③ shift by one

In [ ]:
src, tgt_in, tgt_out = make_batch(pairs)
print("src", tuple(src.shape), " tgt_in", tuple(tgt_in.shape), " tgt_out", tuple(tgt_out.shape))
i = 5   # "good night" -> "buenas noches"
print("src    ", src[i].tolist(), "=", en.decode(src[i].tolist()))
print("tgt_in ", tgt_in[i].tolist(), "=", es.decode(tgt_in[i].tolist()))
print("tgt_out", tgt_out[i].tolist(), "=", es.decode(tgt_out[i].tolist()))

## 6. Training: teacher forcing, padding ignored

The decoder gets the correct Spanish sentence (`tgt_in`) all at once and is scored on the next word at every position (`tgt_out`).
`ignore_index=PAD`: padded positions add nothing to the loss.

In [ ]:
torch.manual_seed(0)
model = Transformer(len(en.itos), len(es.itos))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)  # skip <pad>

src, tgt_in, tgt_out = make_batch(pairs)   # all 11 pairs
losses = []
for epoch in range(1, 201):
    model.train()
    logits = model(src, tgt_in)        # whole target at once
    loss = loss_fn(logits.flatten(0, 1), tgt_out.flatten())
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if epoch in (1, 50, 100, 150, 200):
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.semilogy(range(1, len(losses) + 1), losses)
plt.xlabel("epoch"); plt.ylabel("training loss (log scale)"); plt.grid(alpha=0.3)
plt.show()

### What the loss compares: logits against the correct word

For every position the model outputs a **logit vector** with one score per Spanish word. The target is the **id** of the correct word;
`CrossEntropyLoss` treats that id as a one-hot vector. The cell below computes the same loss by hand with an explicit one-hot vector.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(src, tgt_in)                      # (11 sentences, 5 positions, vocab)
print("logits", tuple(logits.shape), "  one position's logit vector has", logits.shape[-1], "scores")

flat_logits, flat_target = logits.flatten(0, 1), tgt_out.flatten()
keep = flat_target != PAD                            # drop the <pad> positions
one_hot = F.one_hot(flat_target[keep], num_classes=logits.shape[-1]).float()
by_hand = -(one_hot * F.log_softmax(flat_logits[keep], dim=-1)).sum(-1).mean()
print("cross-entropy by hand (one-hot):", round(by_hand.item(), 6))
print("nn.CrossEntropyLoss            :", round(loss_fn(flat_logits, flat_target).item(), 6))

p = F.softmax(logits[5, 0], dim=-1)                  # "good night", first Spanish position
print("most likely first word for 'good night':", es.itos[p.argmax()], f"(p = {p.max():.3f})")

## 7. Translating: one word at a time

Encode the English sentence once, start the Spanish side with `<sos>`, append the best word of the last row, and repeat until `<eos>`.

In [ ]:
@torch.no_grad()
def translate(sentence, max_len=10):
    model.eval()
    src = torch.tensor([en.encode(sentence)])
    memory = model.encode(src)           # ① once per sentence
    ys = [SOS]                           # ②
    for _ in range(max_len):
        logits = model.decode(torch.tensor([ys]), memory, src)  # ③
        nxt = logits[0, -1].argmax().item()   # ④ last row's best
        if nxt == EOS:                   # ⑤
            break
        ys.append(nxt)                   # ⑥
    return es.decode(ys[1:])

In [ ]:
for e, s in pairs:
    out = translate(e)
    print(f"{e:20s} -> {out:20s} {'ok' if out == s else 'expected: ' + s}")

### New combinations of known words

With only 11 pairs the model memorizes: it answers with a sentence it has seen, matching only part of the input.
To translate new sentences it needs thousands of pairs (see the dataset link on the slides).

In [ ]:
for s in ["good night please", "thank you my name is", "see you good morning", "hello how are you"]:
    print(f"{s:24s} -> {translate(s)}")

## 8. The same attention with PyTorch's built-in function

In [ ]:
mha = model.enc[0].attn
x = torch.randn(1, 5, 128)
m = torch.ones(1, 1, 5, 5, dtype=torch.bool)
ours = mha(x, x, m)

B, T, d = x.shape
split = lambda t: t.view(B, -1, mha.h, mha.dk).transpose(1, 2)
out = F.scaled_dot_product_attention(split(mha.q(x)), split(mha.k(x)), split(mha.v(x)), attn_mask=m)
theirs = mha.o(out.transpose(1, 2).reshape(B, T, d))
print("max |ours - built-in| =", (ours - theirs).abs().max().item())

n = lambda mod: sum(p.numel() for p in mod.parameters())
builtin = nn.Transformer(d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2,
                         dim_feedforward=256, batch_first=True)
print("our encoder + decoder layers:", n(model.enc) + n(model.dec))
print("nn.Transformer without its 2 final LayerNorms:", n(builtin) - n(builtin.encoder.norm) - n(builtin.decoder.norm))

## 9. Your turn

* Translate your own sentence (words outside the 11 pairs become `<unk>`).
* Add pairs to `pairs` (the lab asks for about 100), rebuild `en`, `es` and the model, and train again.
* Try `n_layers=1` or `h=1` in `Transformer(...)`, or train for fewer epochs, and compare the translations.

In [ ]:
print(translate("good morning"))